In [1]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\pelli\Documents\TFG_Emociones")

OPENFACE_PATH = BASE_DIR / "archive" / "CMU-MOSEI" / "visuals" / "CMU_MOSEI_VisualOpenFace2.csd"
LABELS_PATH = BASE_DIR / "archive" / "CMU-MOSEI" / "labels" / "CMU_MOSEI_Labels.csd"

print("Visual existe:", OPENFACE_PATH.exists())
print("Labels existe:", LABELS_PATH.exists())
print("Visual:", OPENFACE_PATH)
print("Labels:", LABELS_PATH)

Visual existe: True
Labels existe: True
Visual: C:\Users\pelli\Documents\TFG_Emociones\archive\CMU-MOSEI\visuals\CMU_MOSEI_VisualOpenFace2.csd
Labels: C:\Users\pelli\Documents\TFG_Emociones\archive\CMU-MOSEI\labels\CMU_MOSEI_Labels.csd


In [2]:
import h5py

with h5py.File(OPENFACE_PATH, "r") as f:
    # Ver estructura raíz
    print("Claves raíz:", list(f.keys()))
    
    # Navegar hasta los datos
    def find_data(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} → shape: {obj.shape}, dtype: {obj.dtype}")
    
    f.visititems(find_data)

Claves raíz: ['OpenFace_2']
  OpenFace_2/data/--qXJuDtHPw/features → shape: (1714, 713), dtype: float32
  OpenFace_2/data/--qXJuDtHPw/intervals → shape: (1714, 2), dtype: float64
  OpenFace_2/data/-3g5yACwYnA/features → shape: (4339, 713), dtype: float32
  OpenFace_2/data/-3g5yACwYnA/intervals → shape: (4339, 2), dtype: float64
  OpenFace_2/data/-3nNcZdcdvU/features → shape: (1327, 713), dtype: float32
  OpenFace_2/data/-3nNcZdcdvU/intervals → shape: (1327, 2), dtype: float64
  OpenFace_2/data/-571d8cVauQ/features → shape: (2676, 713), dtype: float32
  OpenFace_2/data/-571d8cVauQ/intervals → shape: (2676, 2), dtype: float64
  OpenFace_2/data/-6rXp3zJ3kc/features → shape: (2254, 713), dtype: float32
  OpenFace_2/data/-6rXp3zJ3kc/intervals → shape: (2254, 2), dtype: float64
  OpenFace_2/data/-9YyBTjo1zo/features → shape: (21092, 713), dtype: float32
  OpenFace_2/data/-9YyBTjo1zo/intervals → shape: (21092, 2), dtype: float64
  OpenFace_2/data/-9y-fZ3swSY/features → shape: (1768, 713), dty

In [3]:
import h5py
import numpy as np

with h5py.File(OPENFACE_PATH, "r") as f:
    first_vid = list(f["OpenFace_2"]["data"].keys())[0]
    sample = f["OpenFace_2"]["data"][first_vid]["features"][:]
    
    print("Shape:", sample.shape)
    print("Primera fila (primeros 20 valores):", sample[0, :20])
    print("Primera fila (valores 700-713):", sample[0, 700:])
    print("Min:", sample.min(), "Max:", sample.max())
    print("¿Hay columnas binarias (0/1)?:", np.all((sample == 0) | (sample == 1), axis=0).sum(), "columnas")

Shape: (1714, 713)
Primera fila (primeros 20 valores): [ 0.00000e+00  3.30000e-02  9.80000e-01  1.00000e+00  8.53220e-02
  1.83282e-01 -9.79351e-01 -7.46660e-02  1.88026e-01 -9.79322e-01
  5.00000e-03  1.87000e-01  1.55400e+02  1.56000e+02  1.57500e+02
  1.59100e+02  1.59800e+02  1.59300e+02  1.57600e+02  1.56100e+02]
Primera fila (valores 700-713): [1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0.]
Min: -336.6 Max: 1353.4
¿Hay columnas binarias (0/1)?: 20 columnas


In [4]:
with h5py.File(OPENFACE_PATH, "r") as f:
    first_vid = list(f["OpenFace_2"]["data"].keys())[0]
    sample = f["OpenFace_2"]["data"][first_vid]["features"][:]

# Layout estándar OpenFace2 (713 features):
# 0        → frame number
# 1        → face confidence
# 2        → success
# 3-5      → pose (Tx, Ty, Tz)
# 6-8      → pose rotation (Rx, Ry, Rz)  <- aquí 3-8 en realidad son 6 valores
# 9-14     → pose (gaze + angles)
# 293-479  → 2D landmarks (x,y) × 68 puntos = 136 valores  
# 480-647  → 3D landmarks × 68 puntos
# 648-712  → AUs

# Verificar bloque AUs (índices 648-712 = 65 valores)
au_block = sample[:, 648:]
print("Shape bloque AU:", au_block.shape)
print("Min/Max bloque AU:", au_block.min(), au_block.max())
print("Primera fila AUs:", au_block[0])

# Verificar columnas binarias en ese bloque
binary_cols = np.all((au_block == 0) | (au_block == 1), axis=0)
print("\nColumnas binarias (presencia) en bloque AU:", binary_cols.sum())
print("Índices binarios:", np.where(binary_cols)[0])

Shape bloque AU: (1714, 65)
Min/Max bloque AU: -8.853 10.431
Primera fila AUs: [ 1.663  0.193  4.759 -2.993  1.55   2.277 -0.49   0.875 -0.254 -3.939
 -0.587  0.519 -0.712  0.142 -1.102 -0.965  0.61  -0.23   0.175  0.037
  0.006  0.091 -0.052  0.108 -0.017  0.032 -0.052  0.02   0.01   0.012
  0.09   0.32   1.78   0.39   0.     1.27   0.21   0.31   0.     0.
  0.     0.     0.     0.     1.12   0.78   1.45   0.     0.     0.
  1.     0.     1.     0.     1.     0.     0.     0.     0.     1.
  0.     1.     0.     0.     0.   ]

Columnas binarias (presencia) en bloque AU: 18
Índices binarios: [47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64]


In [5]:
with h5py.File(OPENFACE_PATH, "r") as f:
    # Coger varios vídeos para tener más datos
    vids = list(f["OpenFace_2"]["data"].keys())[:20]
    samples = []
    for vid in vids:
        s = f["OpenFace_2"]["data"][vid]["features"][:]
        samples.append(s)

all_data = np.vstack(samples)
print("Shape total:", all_data.shape)

# Buscar columnas con rango típico de AU intensidad (0 a 5)
# y mayoría de valores cerca de 0
for i in range(all_data.shape[1]):
    col = all_data[:, i]
    if col.min() >= -0.1 and col.max() <= 6.0 and np.percentile(col, 75) < 3.0:
        binary = np.all((col == 0) | (col == 1))
        print(f"  Col {i:3d} | min={col.min():.3f} max={col.max():.3f} "
              f"mean={col.mean():.3f} | {'BINARIA' if binary else 'intensidad'}")

Shape total: (85172, 713)
  Col   0 | min=0.000 max=0.000 mean=0.000 | BINARIA
  Col   2 | min=0.000 max=0.980 mean=0.855 | intensidad
  Col   3 | min=0.000 max=1.000 mean=0.875 | BINARIA
  Col 678 | min=0.000 max=5.000 mean=0.283 | intensidad
  Col 679 | min=0.000 max=5.000 mean=0.147 | intensidad
  Col 680 | min=0.000 max=4.560 mean=0.513 | intensidad
  Col 681 | min=0.000 max=5.000 mean=0.076 | intensidad
  Col 682 | min=0.000 max=3.320 mean=0.280 | intensidad
  Col 683 | min=0.000 max=5.000 mean=0.572 | intensidad
  Col 684 | min=0.000 max=3.620 mean=0.104 | intensidad
  Col 685 | min=0.000 max=3.920 mean=0.556 | intensidad
  Col 686 | min=0.000 max=4.010 mean=0.289 | intensidad
  Col 687 | min=0.000 max=4.120 mean=0.426 | intensidad
  Col 688 | min=0.000 max=5.000 mean=0.238 | intensidad
  Col 689 | min=0.000 max=5.000 mean=0.713 | intensidad
  Col 690 | min=0.000 max=4.220 mean=0.141 | intensidad
  Col 691 | min=0.000 max=5.000 mean=0.180 | intensidad
  Col 692 | min=0.000 max=5.

In [6]:
import h5py
import numpy as np

# Índices de las AUs en OpenFace2
AU_INTENSITY_COLS = list(range(678, 695))   # 17 AUs intensidad (_r)
AU_PRESENCE_COLS  = list(range(695, 713))   # 18 AUs presencia (_c)
AU_COLS = AU_INTENSITY_COLS + AU_PRESENCE_COLS  # 35 AUs total

def build_visual_dataset_openface(visual_path, labels_path):
    X = []
    y_raw = []
    video_ids = []

    with h5py.File(visual_path, "r") as vf, h5py.File(labels_path, "r") as lf:
        visual_data = vf["OpenFace_2"]["data"]
        label_data  = lf["All Labels"]["data"]

        common_ids = sorted(set(visual_data.keys()) & set(label_data.keys()))
        print(f"Vídeos comunes: {len(common_ids)}")

        for vid in common_ids:
            visual_features = visual_data[vid]["features"][:]
            label_features  = label_data[vid]["features"][:]

            if visual_features.size == 0 or label_features.size == 0:
                continue

            # Extraer solo las 35 AUs
            au_features = visual_features[:, AU_COLS]

            # Limpiar NaN/inf
            au_features = np.nan_to_num(au_features)

            # Agregación temporal: mean + std → vector de 70
            mean_features = np.mean(au_features, axis=0)
            std_features  = np.std(au_features,  axis=0)
            final_features = np.concatenate([mean_features, std_features])

            X.append(final_features)
            y_raw.append(label_features[0])
            video_ids.append(vid)

    return np.array(X), np.array(y_raw), video_ids

X, y_raw, video_ids = build_visual_dataset_openface(OPENFACE_PATH, LABELS_PATH)

print("X shape:", X.shape)          # esperado: (N, 70)
print("y_raw shape:", y_raw.shape)  # esperado: (N, 7)
print("Número de vídeos:", len(video_ids))
print("Primera fila X:", X[0])

Vídeos comunes: 3293
X shape: (3293, 70)
y_raw shape: (3293, 7)
Número de vídeos: 3293
Primera fila X: [0.2290665  0.17238039 1.8066336  0.09630689 0.01495333 0.05225205
 0.04624854 0.606126   0.13567095 0.8796733  0.1487748  0.45002916
 0.10666277 0.08114351 0.42515168 0.476692   0.18932906 0.12368728
 0.2462077  0.64352393 0.9824971  0.01400233 0.01750292 0.
 0.6703617  0.03208868 0.7117853  0.18611436 0.17619604 0.02742124
 0.01866978 0.13418902 0.0630105  0.         0.19311552 0.38770854
 0.4004119  0.814874   0.28441912 0.05464453 0.1395766  0.0893552
 0.3601741  0.15279403 0.4086045  0.21511798 0.39611718 0.18379651
 0.14701377 0.45169744 0.4787653  0.31761798 0.32922447 0.43080094
 0.47895813 0.13113567 0.11750007 0.13113567 0.         0.47008178
 0.17623563 0.45293152 0.38919893 0.38098687 0.16330743 0.13535589
 0.3408553  0.24298185 0.         0.39474285]


In [7]:
# ============================================================
# 1. Crear etiqueta binaria: happy vs not happy
# ============================================================

import numpy as np

happy_scores = y_raw[:, 1]   # columna 1 = happy
y = (happy_scores > 0).astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Distribución [not happy, happy]:", np.bincount(y))

X shape: (3293, 70)
y shape: (3293,)
Distribución [not happy, happy]: [1285 2008]


In [8]:
# ============================================================
# 2. Imports
# ============================================================

import time
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [9]:
# ============================================================
# 3. Definir modelos normal y tiny
# ============================================================

from tensorflow.keras import layers, models

def build_normal_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_tiny_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [10]:
# ============================================================
# 4. Funciones auxiliares
# ============================================================

def compute_class_weights(y_train):
    classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    return dict(zip(classes, weights))


def evaluate_model(model, X_test, y_test):
    start = time.time()
    y_prob = model.predict(X_test, verbose=0).flatten()
    end = time.time()

    y_pred = (y_prob >= 0.5).astype(int)

    latency_ms = ((end - start) / len(X_test)) * 1000

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "latency_ms_per_sample": latency_ms
    }

    try:
        metrics["auc"] = roc_auc_score(y_test, y_prob)
    except ValueError:
        metrics["auc"] = np.nan

    return metrics, y_pred, y_prob

In [11]:
# ============================================================
# 5. K-Fold + selección de características + balanceo
# ============================================================

# Features AU que usa el modelo (definidas en celda 5)
# AU_INTENSITY_COLS = list(range(678, 695))
# AU_PRESENCE_COLS  = list(range(695, 713))
# Recordatorio: el input real son las 70 features (mean+std de 35 AUs)
# tras scaler y selector se reducen a 50

N_SPLITS = 5
K_FEATURES = min(50, X.shape[1])  # X tiene 70 features; probamos top 50
EPOCHS = 40
BATCH_SIZE = 32
RANDOM_STATE = 42

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

best_tiny_model = None
best_tiny_f1 = -1
best_fold = None

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n==============================")
    print(f"Fold {fold}/{N_SPLITS}")
    print(f"==============================")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # --------------------------------------------------------
    # Normalización SOLO con train para evitar data leakage
    # --------------------------------------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------
    # Selección de características SOLO con train
    # --------------------------------------------------------
    selector = SelectKBest(score_func=f_classif, k=K_FEATURES)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)

    # --------------------------------------------------------
    # Balanceo de clases SOLO con train
    # --------------------------------------------------------
    class_weight = compute_class_weights(y_train)

    input_dim = X_train_selected.shape[1]

    model_builders = {
        "normal_model": build_normal_model,
        "tiny_model": build_tiny_model
    }

    for model_name, builder in model_builders.items():
        print(f"Entrenando {model_name}...")

        model = builder(input_dim)

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )

        model.fit(
            X_train_selected,
            y_train,
            validation_split=0.2,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=0
        )

        metrics, y_pred, y_prob = evaluate_model(
            model,
            X_test_selected,
            y_test
        )

        row = {
            "fold": fold,
            "model": model_name,
            "num_features": input_dim,
            "num_parameters": model.count_params(),
            **metrics
        }

        results.append(row)

        print(model_name, row)

        if model_name == "tiny_model" and metrics["f1_score"] > best_tiny_f1:
            best_tiny_f1 = metrics["f1_score"]
            best_tiny_model = model
            best_fold = fold
            best_scaler = scaler
            best_selector = selector

results_df = pd.DataFrame(results)
results_df


Fold 1/5
Entrenando normal_model...
normal_model {'fold': 1, 'model': 'normal_model', 'num_features': 50, 'num_parameters': 54273, 'accuracy': 0.6388467374810318, 'precision': 0.777027027027027, 'recall': 0.572139303482587, 'f1_score': 0.6590257879656161, 'latency_ms_per_sample': 0.12981439396535502, 'auc': 0.7221044582534797}
Entrenando tiny_model...
tiny_model {'fold': 1, 'model': 'tiny_model', 'num_features': 50, 'num_parameters': 2177, 'accuracy': 0.6282245827010622, 'precision': 0.7415384615384616, 'recall': 0.599502487562189, 'f1_score': 0.6629986244841816, 'latency_ms_per_sample': 0.10782236032674092, 'auc': 0.7041688444934859}

Fold 2/5
Entrenando normal_model...
normal_model {'fold': 2, 'model': 'normal_model', 'num_features': 50, 'num_parameters': 54273, 'accuracy': 0.6676783004552352, 'precision': 0.7592067988668555, 'recall': 0.6666666666666666, 'f1_score': 0.7099337748344371, 'latency_ms_per_sample': 0.11532693784768737, 'auc': 0.7300753044117931}
Entrenando tiny_model...

,fold,model,num_features,num_parameters,accuracy,precision,recall,f1_score,latency_ms_per_sample,auc
0,1,normal_model,50,54273,0.638847,0.777027,0.572139,0.659026,0.129814,0.722104
1,1,tiny_model,50,2177,0.628225,0.741538,0.599502,0.662999,0.107822,0.704169
2,2,normal_model,50,54273,0.667678,0.759207,0.666667,0.709934,0.115327,0.730075
3,2,tiny_model,50,2177,0.676783,0.750663,0.703980,0.726573,0.106221,0.731401
4,3,normal_model,50,54273,0.654021,0.737705,0.671642,0.703125,0.115431,0.704512
5,3,tiny_model,50,2177,0.666161,0.770833,0.644279,0.701897,0.107737,0.723619
6,4,normal_model,50,54273,0.662614,0.742547,0.683292,0.711688,0.120094,0.719126
7,4,tiny_model,50,2177,0.650456,0.755224,0.630923,0.687500,0.107903,0.721339
8,5,normal_model,50,54273,0.662614,0.757925,0.655860,0.703209,0.118541,0.724618
9,5,tiny_model,50,2177,0.655015,0.740331,0.668329,0.702490,0.106384,0.707870


In [12]:
# ============================================================
# 6. Resumen de resultados
# ============================================================

summary = results_df.groupby("model").agg({
    "accuracy": ["mean", "std"],
    "precision": ["mean", "std"],
    "recall": ["mean", "std"],
    "f1_score": ["mean", "std"],
    "auc": ["mean", "std"],
    "num_parameters": "mean",
    "latency_ms_per_sample": ["mean", "std"]
})

summary

accuracy           precision              recall            \
                  mean       std      mean       std      mean       std   
model                                                                      
normal_model  0.657155  0.011351  0.754882  0.015537  0.649920  0.044584   
tiny_model    0.655328  0.018276  0.751718  0.012370  0.649403  0.039353   

              f1_score                 auc           num_parameters  \
                  mean       std      mean       std           mean   
model                                                                 
normal_model  0.697396  0.021796  0.720087  0.009590        54273.0   
tiny_model    0.696292  0.023305  0.717680  0.011355         2177.0   

             latency_ms_per_sample            
                              mean       std  
model                                         
normal_model              0.119842  0.005938  
tiny_model                0.107213  0.000836

In [13]:
# ============================================================
# 7. Guardar resultados
# ============================================================

from pathlib import Path

RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = BASE_DIR / "models"

RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

results_df.to_csv(RESULTS_DIR / "kfold_results_happy_vs_not_happy_OpenFace2.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary_happy_vs_not_happy_OpenFace2.csv")

print("Resultados guardados en:", RESULTS_DIR)

Resultados guardados en: C:\Users\pelli\Documents\TFG_Emociones\results


In [14]:
# ============================================================
# 8. Guardar mejor modelo tiny
# ============================================================

best_model_path = MODELS_DIR / f"best_tiny_model_fold_{best_fold}_OpenFace2.keras"
best_tiny_model.save(best_model_path)

print("Mejor fold tiny:", best_fold)
print("Mejor F1 tiny:", best_tiny_f1)
print("Modelo guardado en:", best_model_path)

Mejor fold tiny: 2
Mejor F1 tiny: 0.7265725288831836
Modelo guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model_fold_2_OpenFace2.keras


In [15]:
# ============================================================
# 9. Exportar mejor modelo tiny a TensorFlow Lite
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)
tflite_model = converter.convert()

tflite_path = MODELS_DIR / "best_tiny_model_OpenFace2.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Modelo TFLite guardado en:", tflite_path)

INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmp9y2zdkjh\assets


INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmp9y2zdkjh\assets


Saved artifact at 'C:\Users\pelli\AppData\Local\Temp\tmp9y2zdkjh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2768658347792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658339920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658349136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658342992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658345488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658348560: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo TFLite guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model_OpenFace2.tflite


In [16]:
# ============================================================
# 10. Exportar versión cuantizada opcional
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

tflite_quant_path = MODELS_DIR / "best_tiny_model_quantized_OpenFace2.tflite"

with open(tflite_quant_path, "wb") as f:
    f.write(tflite_quant_model)

print("Modelo TFLite cuantizado guardado en:", tflite_quant_path)

INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmp0ks6pztg\assets


INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmp0ks6pztg\assets


Saved artifact at 'C:\Users\pelli\AppData\Local\Temp\tmp0ks6pztg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2768658347792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658339920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658349136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658342992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658345488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2768658348560: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo TFLite cuantizado guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model_quantized_OpenFace2.tflite


In [17]:
# ============================================================
# 11. Guardar scaler y selector del mejor fold
# ============================================================
import pickle

with open(MODELS_DIR / "best_scaler_OpenFace2.pkl", "wb") as f:
    pickle.dump(best_scaler, f)

with open(MODELS_DIR / "best_selector_OpenFace2.pkl", "wb") as f:
    pickle.dump(best_selector, f)

# Mostrar qué índices de features seleccionó
selected_indices = best_selector.get_support(indices=True)
print("Scaler y selector guardados.")
print(f"Índices de las {len(selected_indices)} features seleccionadas:", selected_indices.tolist())

Scaler y selector guardados.
Índices de las 50 features seleccionadas: [0, 2, 3, 4, 5, 6, 8, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 66, 67, 69]


In [18]:
import json
import numpy as np

# Exportar scaler
scaler_mean = best_scaler.mean_.tolist()
scaler_scale = best_scaler.scale_.tolist()

with open(MODELS_DIR / "best_scaler_mean.json", "w") as f:
    json.dump(scaler_mean, f)

with open(MODELS_DIR / "best_scaler_scale.json", "w") as f:
    json.dump(scaler_scale, f)

# Exportar índices del selector
selected_indices = best_selector.get_support(indices=True).tolist()

with open(MODELS_DIR / "best_selector_indices.json", "w") as f:
    json.dump(selected_indices, f)

print("Exportado scaler mean:", len(scaler_mean), "valores")
print("Exportado scaler scale:", len(scaler_scale), "valores")
print("Exportado selector indices:", selected_indices)

Exportado scaler mean: 70 valores
Exportado scaler scale: 70 valores
Exportado selector indices: [0, 2, 3, 4, 5, 6, 8, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 66, 67, 69]
